# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ubaidrees/flyrank-ml-tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row = one content item (content_hash_id), summarized over a 90-day feature window (2026-01-01 to 2026-03-31), with a separate 30-day target window (2026-04-01 to 2026-04-30) used only to build the label — never mixed into the features. Source table: fact_content_daily_performance, joined to dim_clients to check tracking availability before including any client's rows. Grain and window are verified with real queries in Section 3.
Features (5), each with an availability check:

impressions_prev90 — sum of gsc_impressions, Jan 1–Mar 31 — known at decision time; window ends before the target period starts.

clicks_prev90 — sum of gsc_clicks, same window — same reasoning.

avg_position_prev90 — average gsc_avg_position, same window — same reasoning.

active_days_prev90 — count of days with impressions > 0 in the window — purely historical, no future data touched.

days_since_last_activity — days between the content's last active day in the window and March 31 — a recency signal, still fully historical.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Label: is_drop_imminent = 1 if total GSC clicks in the April target window are ≥20% below total clicks in the prior-90-day feature window.

Excluded: rows from clients where has_gsc_access is not TRUE, or whose gsc_data_start is later than 2026-01-01 — meaning their 90-day feature window isn't fully covered by real tracking. Including them would risk mistaking "not tracked yet" for "no traffic."

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb, os, getpass
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily  = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_clients = f"read_parquet('{REL}/dim_clients.parquet')"

# ---- Query 1: Grain check ----
# Claim: after aggregating to one row per content item, content_hash_id is unique
grain_check = con.sql(f"""
    WITH agg AS (
        SELECT content_hash_id, COUNT(*) AS n
        FROM {fact_daily}
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
        GROUP BY content_hash_id
    )
    SELECT COUNT(*) AS content_items_seen
    FROM agg
""").df()
print("Grain check (should show real count, no duplicate content_hash_id after our own GROUP BY):")
print(grain_check)

# ---- Query 2: Row count + date span ----
span_check = con.sql(f"""
    SELECT COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date
    FROM {fact_daily}
    WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
""").df()
print("\nRow count + date span for the feature window:")
print(span_check)

# ---- Query 3: Availability check, filtered with IS TRUE ----
before_filter = con.sql(f"""
    SELECT COUNT(*) AS rows_before
    FROM {fact_daily} f
    WHERE f.report_date BETWEEN '2026-01-01' AND '2026-03-31'
""").fetchone()[0]

after_filter = con.sql(f"""
    SELECT COUNT(*) AS rows_after
    FROM {fact_daily} f
    JOIN {dim_clients} c ON f.client_hash_id = c.client_hash_id
    WHERE f.report_date BETWEEN '2026-01-01' AND '2026-03-31'
      AND c.has_gsc_access IS TRUE
      AND c.gsc_data_start <= '2026-01-01'
""").fetchone()[0]

print(f"\nAvailability check:")
print(f"Rows before filter: {before_filter:,}")
print(f"Rows after IS TRUE + tracking-start filter: {after_filter:,}")
print(f"Rows dropped: {before_filter - after_filter:,} ({(1 - after_filter/before_filter)*100:.1f}%)")

In [ ]:
# ---- THE TRAP: build the honest feature/label frame first ----
data = con.sql(f"""
    WITH bounds AS (SELECT DATE '2026-03-31' AS cutoff),
    feat AS (
        SELECT f.content_hash_id,
               SUM(f.gsc_impressions) AS impressions_prev90,
               SUM(f.gsc_clicks) AS clicks_prev90,
               AVG(f.gsc_avg_position) AS avg_position_prev90,
               COUNT(*) FILTER (WHERE f.gsc_impressions > 0) AS active_days_prev90
        FROM {fact_daily} f, bounds b
        WHERE f.report_date BETWEEN '2026-01-01' AND '2026-03-31'
        GROUP BY f.content_hash_id
    ),
    target AS (
        SELECT content_hash_id, SUM(gsc_clicks) AS clicks_next30
        FROM {fact_daily}
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY content_hash_id
    )
    SELECT feat.*, target.clicks_next30
    FROM feat JOIN target USING (content_hash_id)
""").df()

data['is_drop_imminent'] = (data['clicks_next30'] < 0.8 * (data['clicks_prev90'])).astype(int)
data = data.dropna()

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_prev90', 'clicks_prev90', 'avg_position_prev90', 'active_days_prev90']
X, y = data[honest_features], data['is_drop_imminent']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"HONEST score (no leakage): AUC = {honest_auc:.3f}")

# ---- Now sneak in the leak: clicks_next30 is literally used to build the label ----
data['leaked_feature'] = data['clicks_next30']  # <-- this IS the label's own source data
leaky_features = honest_features + ['leaked_feature']
X2 = data[leaky_features]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y, test_size=0.25, random_state=42, stratify=y)
model2 = RandomForestClassifier(n_estimators=200, random_state=42).fit(X2_tr, y2_tr)
leaky_auc = roc_auc_score(y2_te, model2.predict_proba(X2_te)[:, 1])
print(f"LEAKY score (with target-derived feature): AUC = {leaky_auc:.3f}")

print(f"\nJump from leakage: {leaky_auc - honest_auc:+.3f}")
print("Deleting 'leaked_feature' now — the honest AUC above is the real number to report.")

In [ ]:
# Re-run the "honest" model WITHOUT clicks_prev90, since it's baked into the label's own formula
truly_honest_features = ['impressions_prev90', 'avg_position_prev90', 'active_days_prev90']
X3 = data[truly_honest_features]
X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y, test_size=0.25, random_state=42, stratify=y)
model3 = RandomForestClassifier(n_estimators=200, random_state=42).fit(X3_tr, y3_tr)
truly_honest_auc = roc_auc_score(y3_te, model3.predict_proba(X3_te)[:, 1])
print(f"TRULY honest score (label-independent features only): AUC = {truly_honest_auc:.3f}")

Second leak found beyond the required trap: the "honest" model (AUC 0.972) still included clicks_prev90, which is the same quantity the label's threshold is built from (clicks_next30 < 0.8 × clicks_prev90). Removing it dropped AUC to 0.866 — a real result, not formula-assisted. 0.866 is the number I'd actually report, not 0.972. This shows leakage doesn't only come from obviously target-derived columns; it can also come from a feature that's structurally coupled to how the label itself is calculated.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
Limitation: this slice only includes clients with has_gsc_access = TRUE and gsc_data_start on or before 2026-01-01, so their full 90-day feature window is backed by real tracking. This means the results here don't generalize to newly onboarded clients — anyone whose tracking started mid-window is excluded, so this contract can't speak to "how does a brand-new client's content behave," only to established ones with a full quarter of history.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.